# Notebook 2 of 2 — Build the Silver layer

**Your job:** read your Bronze tables, find out what is wrong with the data, and publish **trustworthy Silver** tables, with the bad rows kept in a rejected table so nothing disappears silently.

> **Learning rule:** AI tools are allowed here. Understanding is not optional. You will explain every transformation in your own words in `OBSERVATION_REPORT.md`.

### The naming contract

| Output | ADLS path (container `lakehouse`) | Unity Catalog table |
|---|---|---|
| Clean trips | `silver/taxi_trips/` | `<catalog>.silver.taxi_trips` |
| Rejected trips | `silver/taxi_trips_rejected/` | `<catalog>.silver.taxi_trips_rejected` |
| Clean weather | `silver/weather_daily/` | `<catalog>.silver.weather_daily` |

External Delta tables again, same storage account and container as Bronze. Silver writes use `overwrite`, so the notebook can be re-run safely.

In [0]:
# ---- Fill in the two values marked <...> (same values as notebook 1) ----------
STORAGE_ACCOUNT = "stnycweatherdev001"
CATALOG         = "dbw_nyc_weather_dev"

# ---- Do NOT change these ------------------------------------------------------
CONTAINER = "lakehouse"
BASE      = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
MONTHS    = ["2025-08", "2025-09", "2025-10"]

# The 20 columns exactly as the NYC TLC publishes them (Bronze keeps these names)
SRC_COLS = [
    "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count", "trip_distance",
    "RatecodeID", "store_and_fwd_flag", "PULocationID", "DOLocationID", "payment_type", "fare_amount",
    "extra", "mta_tax", "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount",
    "congestion_surcharge", "Airport_fee", "cbd_congestion_fee",
]

---
## Step 1 — Look before you clean (profile Bronze)

Write queries that answer these four questions. Put the numbers in report §5.

1. How many rows in total, and how many nulls per column?
2. What is the earliest and latest pickup? How many pickups fall **outside the month of the file they came from**?
3. What values does `payment_type` take, and how many rows each?
4. How many trips have dropoff ≤ pickup, distance ≤ 0, fare ≤ 0?

**Hint:** one pass with `F.sum(F.col(c).isNull().cast("int"))` over `SRC_COLS` gives all null counts at once.

**Pause and observe →** Several columns show the *exact same* huge null count. Which columns, and which `payment_type` value has that same count? What does that tell you about those rows? Would deleting them be right? Decide, and defend it in report §6.

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, min as spark_min, max as spark_max, date_format, lit

# Read the Bronze taxi trips table
df_bronze = spark.table(f"{CATALOG}.bronze.taxi_trips")

print(f"=== 1. Total Rows & Nulls per Column ===")
total_rows = df_bronze.count()
print(f"Total Rows: {total_rows}")

# Null counts across all 20 source columns
null_exprs = [spark_sum(col(c).isNull().cast("int")).alias(c) for c in SRC_COLS]
display(df_bronze.select(null_exprs))


print(f"\n=== 2. Earliest/Latest Pickups & Out-of-Month Count ===")
# Earliest and latest pickup timestamps
display(df_bronze.select(spark_min("tpep_pickup_datetime").alias("earliest_pickup"), spark_max("tpep_pickup_datetime").alias("latest_pickup")))

# Count pickups falling outside their source_month
out_of_month_count = df_bronze.filter(
    date_format(col("tpep_pickup_datetime"), "yyyy-MM") != col("source_month")
).count()
print(f"Pickups outside source_month: {out_of_month_count}")


print(f"\n=== 3. Payment Type Breakdown ===")
display(df_bronze.groupBy("payment_type").count().orderBy("count", ascending=False))


print(f"\n=== 4. Anomaly Counts (Dropoff <= Pickup, Distance <= 0, Fare <= 0) ===")
anomalies = df_bronze.select(
    spark_sum((col("tpep_dropoff_datetime") <= col("tpep_pickup_datetime")).cast("int")).alias("bad_dropoff_time"),
    spark_sum((col("trip_distance") <= 0).cast("int")).alias("non_positive_distance"),
    spark_sum((col("fare_amount") <= 0).cast("int")).alias("non_positive_fare")
)
display(anomalies)

=== 1. Total Rows & Nulls per Column ===
Total Rows: 12253805


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,0,0,2944316,0,2944316,2944316,0,0,0,0,0,0,0,0,0,0,2944316,2944316,0



=== 2. Earliest/Latest Pickups & Out-of-Month Count ===


earliest_pickup,latest_pickup
2009-01-01T12:52:15.000Z,2025-11-01T00:32:12.000Z


Pickups outside source_month: 37

=== 3. Payment Type Breakdown ===


payment_type,count
1,7777780
0,2944316
2,1141826
4,308046
3,81837



=== 4. Anomaly Counts (Dropoff <= Pickup, Distance <= 0, Fare <= 0) ===


bad_dropoff_time,non_positive_distance,non_positive_fare
172980,355266,840442


---
## Step 2 — Split off the rejects

A row is **rejected** if it breaks any rule below. Each rejected row gets exactly **one** `reject_reason` (the **first** rule it fails, in this order):

| # | `reject_reason` (exact string) | Rejected when |
|---|---|---|
| 1 | `pickup_outside_source_month` | the pickup month ≠ `source_month` |
| 2 | `dropoff_not_after_pickup` | dropoff ≤ pickup |
| 3 | `duration_over_6h` | trip lasts more than 360 minutes |
| 4 | `distance_invalid` | `trip_distance` ≤ 0 or ≥ 100 miles |
| 5 | `fare_invalid` | `fare_amount` ≤ 0 or ≥ 500 |

Write rejected rows to `silver/taxi_trips_rejected/`: **all Bronze columns unchanged**, plus `reject_reason` (string) and `rejected_at` (timestamp).

**Hints**
- A chain of `F.when(...).when(...)` returns the first match, which is exactly "first failing rule".
- Keep the original column names and types here. This table is evidence for someone investigating.
- Count rejected rows per reason and keep **one example row per reason** for the report.

**Pause and observe →** Why does the table keep the *original* values instead of cleaned ones? Why one reason per row? Report §6.

In [0]:
from pyspark.sql.functions import col, when, current_timestamp, date_format, lit, unix_timestamp

#  Read the bronze table
df_bronze = spark.table(f"{CATALOG}.bronze.taxi_trips")

#  Evaluate rows against the strict priority rule order
df_evaluated = df_bronze.withColumn(
    "reject_reason",
    when(date_format(col("tpep_pickup_datetime"), "yyyy-MM") != col("source_month"), lit("pickup_outside_source_month"))
    .when(col("tpep_dropoff_datetime") <= col("tpep_pickup_datetime"), lit("dropoff_not_after_pickup"))
    .when((unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) > 360 * 60, lit("duration_over_6h"))
    .when((col("trip_distance") <= 0) | (col("trip_distance") >= 100), lit("distance_invalid"))
    .when((col("fare_amount") <= 0) | (col("fare_amount") >= 500), lit("fare_invalid"))
    .otherwise(None)
)

#  Filter for rejected rows and add the rejected_at timestamp
df_rejected = (
    df_evaluated
    .filter(col("reject_reason").isNotNull())
    .withColumn("rejected_at", current_timestamp())
)

#  Write rejected rows to silver/taxi_trips_rejected/
silver_rejected_path = f"{BASE}/silver/taxi_trips_rejected/"

(
    df_rejected.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .partitionBy("source_month")
    .save(silver_rejected_path)
)

print("--- Rejects Partition Written Successfully ---")

#  Count rejected rows per reason
print("\nRejected count per reason:")
display(df_rejected.groupBy("reject_reason").count().orderBy("count", ascending=False))

--- Rejects Partition Written Successfully ---

Rejected count per reason:


reject_reason,count
fare_invalid,766477
distance_invalid,351499
dropoff_not_after_pickup,172980
duration_over_6h,3490
pickup_outside_source_month,37


---
## Step 3 — Clean the good rows into `silver.taxi_trips`

Take the rows that passed Step 2, **remove exact duplicates** (all 20 source columns equal; count how many you removed into a variable named `DUPLICATES_REMOVED`), and shape them exactly like this:

| Bronze column | Silver column | Silver type | Note |
|---|---|---|---|
| `VendorID` | `vendor_id` | int | |
| `tpep_pickup_datetime` | `pickup_ts` | timestamp | |
| `tpep_dropoff_datetime` | `dropoff_ts` | timestamp | |
| `passenger_count` | `passenger_count` | int | outside 1–6 → `NULL` (keep the trip, fix the value) |
| `trip_distance` | `trip_distance_miles` | decimal(8,2) | |
| `RatecodeID` | `rate_code_id` | int | |
| `store_and_fwd_flag` | `store_and_fwd_flag` | boolean | `Y` → true, `N` → false, anything else → null |
| `PULocationID` | `pickup_location_id` | int | |
| `DOLocationID` | `dropoff_location_id` | int | |
| `payment_type` | `payment_type_id` | int | |
| *(new)* | `payment_type_desc` | string | 0 Flex fare · 1 Credit card · 2 Cash · 3 No charge · 4 Dispute · 5 Unknown · 6 Voided trip · else `Other` |
| `fare_amount`, `extra`, `mta_tax`, `tip_amount`, `tolls_amount`, `improvement_surcharge`, `total_amount`, `congestion_surcharge`, `cbd_congestion_fee` | same names | decimal(10,2) | money is never a float |
| `Airport_fee` | `airport_fee` | decimal(10,2) | snake_case |
| *(new)* | `pickup_date` | date | from the pickup timestamp |
| *(new)* | `pickup_hour` | int | 0–23 |
| *(new)* | `trip_duration_min` | decimal(8,2) | (dropoff − pickup) in minutes |
| `source_month` | `source_month` | string | carried over |
| *(new)* | `silver_loaded_at` | timestamp | |

Columns not in this table (`source_file`, `ingested_at`) are **not** carried to Silver. The column order above is the order to use.

**Hints**
- `dropDuplicates(SRC_COLS)` removes exact duplicates; the difference of two counts tells you how many.
- `.cast("decimal(10,2)")` for money. An unknown payment type must become `Other`, never an error.
- `F.to_date`, `F.hour`, `F.unix_timestamp` cover the derived columns.
- Write with `overwrite` to `silver/taxi_trips/`, then `printSchema()` and count per `source_month`.

**Pause and observe →** Why decimal for money? Why set an out-of-range passenger count to `NULL` rather than reject the trip, when you *rejected* a ≤ 0 fare? Report §7 asks you to fill a before/after schema table.

In [0]:
from pyspark.sql.functions import col, when, current_timestamp, to_date, hour, unix_timestamp, lit, date_format

# 1. Start with rows that passed Step 2 (not rejected)
df_bronze = spark.table(f"{CATALOG}.bronze.taxi_trips")

# Re-apply the same rejection condition to filter out the clean rows
df_evaluated = df_bronze.withColumn(
    "reject_reason",
    when(date_format(col("tpep_pickup_datetime"), "yyyy-MM") != col("source_month"), lit("pickup_outside_source_month"))
    .when(col("tpep_dropoff_datetime") <= col("tpep_pickup_datetime"), lit("dropoff_not_after_pickup"))
    .when((unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) > 360 * 60, lit("duration_over_6h"))
    .when((col("trip_distance") <= 0) | (col("trip_distance") >= 100), lit("distance_invalid"))
    .when((col("fare_amount") <= 0) | (col("fare_amount") >= 500), lit("fare_invalid"))
    .otherwise(None)
)

df_clean_candidate = df_evaluated.filter(col("reject_reason").isNull())

# 2. Track duplicate removal count
initial_clean_count = df_clean_candidate.count()
df_deduplicated = df_clean_candidate.dropDuplicates(SRC_COLS)
final_clean_count = df_deduplicated.count()

DUPLICATES_REMOVED = initial_clean_count - final_clean_count
print(f"Exact duplicates removed: {DUPLICATES_REMOVED}")

# 3. Shape, cast, and map columns to the Silver specification
df_silver_taxi = (
    df_deduplicated
    # Rename & Cast Base Columns
    .withColumn("vendor_id", col("VendorID").cast("int"))
    .withColumn("pickup_ts", col("tpep_pickup_datetime").cast("timestamp"))
    .withColumn("dropoff_ts", col("tpep_dropoff_datetime").cast("timestamp"))
    # passenger_count: outside 1-6 -> NULL
    .withColumn("passenger_count", when((col("passenger_count") >= 1) & (col("passenger_count") <= 6), col("passenger_count").cast("int")).otherwise(None))
    .withColumn("trip_distance_miles", col("trip_distance").cast("decimal(8,2)"))
    .withColumn("rate_code_id", col("RatecodeID").cast("int"))
    # store_and_fwd_flag: Y -> True, N -> False, else -> null (using Python/Spark lit(True/False))
    .withColumn("store_and_fwd_flag", when(col("store_and_fwd_flag") == "Y", lit(True)).when(col("store_and_fwd_flag") == "N", lit(False)).otherwise(None))
    .withColumn("pickup_location_id", col("PULocationID").cast("int"))
    .withColumn("dropoff_location_id", col("DOLocationID").cast("int"))
    .withColumn("payment_type_id", col("payment_type").cast("int"))
    # payment_type_desc mapping
    .withColumn("payment_type_desc", 
        when(col("payment_type") == 0, "Flex fare")
        .when(col("payment_type") == 1, "Credit card")
        .when(col("payment_type") == 2, "Cash")
        .when(col("payment_type") == 3, "No charge")
        .when(col("payment_type") == 4, "Dispute")
        .when(col("payment_type") == 5, "Unknown")
        .when(col("payment_type") == 6, "Voided trip")
        .otherwise("Other")
    )
    # Monetary amounts -> decimal(10,2)
    .withColumn("fare_amount", col("fare_amount").cast("decimal(10,2)"))
    .withColumn("extra", col("extra").cast("decimal(10,2)"))
    .withColumn("mta_tax", col("mta_tax").cast("decimal(10,2)"))
    .withColumn("tip_amount", col("tip_amount").cast("decimal(10,2)"))
    .withColumn("tolls_amount", col("tolls_amount").cast("decimal(10,2)"))
    .withColumn("improvement_surcharge", col("improvement_surcharge").cast("decimal(10,2)"))
    .withColumn("total_amount", col("total_amount").cast("decimal(10,2)"))
    .withColumn("congestion_surcharge", col("congestion_surcharge").cast("decimal(10,2)"))
    .withColumn("cbd_congestion_fee", col("cbd_congestion_fee").cast("decimal(10,2)"))
    .withColumn("airport_fee", col("Airport_fee").cast("decimal(10,2)"))
    # Derived features
    .withColumn("pickup_date", to_date(col("pickup_ts")))
    .withColumn("pickup_hour", hour(col("pickup_ts")))
    .withColumn("trip_duration_min", ((unix_timestamp(col("dropoff_ts")) - unix_timestamp(col("pickup_ts"))) / 60.0).cast("decimal(8,2)"))
    .withColumn("source_month", col("source_month"))
    .withColumn("silver_loaded_at", current_timestamp())
    # Select only the target columns in the exact order requested
    .select(
        "vendor_id", "pickup_ts", "dropoff_ts", "passenger_count", "trip_distance_miles",
        "rate_code_id", "store_and_fwd_flag", "pickup_location_id", "dropoff_location_id",
        "payment_type_id", "payment_type_desc", "fare_amount", "extra", "mta_tax",
        "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount",
        "congestion_surcharge", "cbd_congestion_fee", "airport_fee", "pickup_date",
        "pickup_hour", "trip_duration_min", "source_month", "silver_loaded_at"
    )
)

# 4. Write to Silver Delta table partitioned by source_month
silver_taxi_path = f"{BASE}/silver/taxi_trips/"

(
    df_silver_taxi.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", "source_month in ('2025-08', '2025-09', '2025-10')")
    .option("mergeSchema", "true")
    .partitionBy("source_month")
    .save(silver_taxi_path)
)

print("--- Silver Clean Taxi Table Written Successfully ---")
df_silver_taxi.printSchema()

print("\nRow count per source_month in Silver:")
display(spark.read.format("delta").load(silver_taxi_path).groupBy("source_month").count().orderBy("source_month"))

Exact duplicates removed: 0
--- Silver Clean Taxi Table Written Successfully ---
root
 |-- vendor_id: integer (nullable = true)
 |-- pickup_ts: timestamp (nullable = true)
 |-- dropoff_ts: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance_miles: decimal(8,2) (nullable = true)
 |-- rate_code_id: integer (nullable = true)
 |-- store_and_fwd_flag: boolean (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- dropoff_location_id: integer (nullable = true)
 |-- payment_type_id: integer (nullable = true)
 |-- payment_type_desc: string (nullable = false)
 |-- fare_amount: decimal(10,2) (nullable = true)
 |-- extra: decimal(10,2) (nullable = true)
 |-- mta_tax: decimal(10,2) (nullable = true)
 |-- tip_amount: decimal(10,2) (nullable = true)
 |-- tolls_amount: decimal(10,2) (nullable = true)
 |-- improvement_surcharge: decimal(10,2) (nullable = true)
 |-- total_amount: decimal(10,2) (nullable = true)
 |-- congestion_surcharge: dec

source_month,count
2025-08,3179661
2025-09,3837040
2025-10,3942621


---
## Step 4 — Silver weather

Read `bronze.weather_daily` and write `silver/weather_daily/`:

| Bronze column | Silver column | Silver type | Note |
|---|---|---|---|
| `time` | `weather_date` | date | one row per date, drop duplicates on the date |
| `temperature_2m_max` | `temp_max_c` | decimal(5,1) | |
| `temperature_2m_min` | `temp_min_c` | decimal(5,1) | |
| `precipitation_sum` | `precip_mm` | decimal(6,1) | |
| `snowfall_sum` | `snowfall_cm` | decimal(6,1) | |
| `wind_speed_10m_max` | `wind_max_kmh` | decimal(6,1) | |
| *(new)* | `is_rainy` | boolean | true when precipitation is **1.0 mm or more** |
| *(new)* | `silver_loaded_at` | timestamp | |

**Pause and observe →** Why encode the units (`_c`, `_mm`, `_cm`, `_kmh`) in the column names? Report §7.

In [0]:
from pyspark.sql.functions import col, when, current_timestamp, to_date, lit

# 1. Read the Bronze weather table
df_bronze_weather = spark.table(f"{CATALOG}.bronze.weather_daily")

# 2. Transform, cast, and map columns to Silver weather specifications
df_silver_weather = (
    df_bronze_weather
    .withColumn("weather_date", to_date(col("time")))
    .withColumn("temp_max_c", col("temperature_2m_max").cast("decimal(5,1)"))
    .withColumn("temp_min_c", col("temperature_2m_min").cast("decimal(5,1)"))
    .withColumn("precip_mm", col("precipitation_sum").cast("decimal(6,1)"))
    .withColumn("snowfall_cm", col("snowfall_sum").cast("decimal(6,1)"))
    .withColumn("wind_max_kmh", col("wind_speed_10m_max").cast("decimal(6,1)"))
    .withColumn("is_rainy", when(col("precipitation_sum") >= 1.0, lit(True)).otherwise(lit(False)))
    .withColumn("silver_loaded_at", current_timestamp())
    # Drop duplicates on the weather date to ensure 1 row per date
    .dropDuplicates(["weather_date"])
    # Select columns in the exact order requested
    .select(
        "weather_date", "temp_max_c", "temp_min_c", "precip_mm",
        "snowfall_cm", "wind_max_kmh", "is_rainy", "silver_loaded_at"
    )
)

# 3. Write to Silver weather Delta table
silver_weather_path = f"{BASE}/silver/weather_daily/"

(
    df_silver_weather.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .save(silver_weather_path)
)

print("--- Silver Weather Table Written Successfully ---")
df_silver_weather.printSchema()
print(f"Total rows in Silver weather: {df_silver_weather.count()}")
display(df_silver_weather.limit(5))

--- Silver Weather Table Written Successfully ---
root
 |-- weather_date: date (nullable = true)
 |-- temp_max_c: decimal(5,1) (nullable = true)
 |-- temp_min_c: decimal(5,1) (nullable = true)
 |-- precip_mm: decimal(6,1) (nullable = true)
 |-- snowfall_cm: decimal(6,1) (nullable = true)
 |-- wind_max_kmh: decimal(6,1) (nullable = true)
 |-- is_rainy: boolean (nullable = false)
 |-- silver_loaded_at: timestamp (nullable = false)

Total rows in Silver weather: 92


weather_date,temp_max_c,temp_min_c,precip_mm,snowfall_cm,wind_max_kmh,is_rainy,silver_loaded_at
2025-08-01,23.0,17.8,1.0,0.0,16.1,true,2026-11-18T15:52:43.059Z
2025-08-02,25.7,17.1,0.0,0.0,10.1,false,2026-11-18T15:52:43.059Z
2025-08-03,28.1,16.5,0.0,0.0,11.1,false,2026-11-18T15:52:43.059Z
2025-08-04,33.2,17.0,0.0,0.0,14.6,false,2026-11-18T15:52:43.059Z
2025-08-05,31.0,19.9,0.0,0.0,9.7,false,2026-11-18T15:52:43.059Z


---
## Step 5 — Register, reconcile and sanity-check

1. Create schema `silver` and register the three **external** tables from the contract table.
2. Run the check cell below (given). Fix every ❌. It also reconciles **bronze = silver + rejected + duplicates removed**.
3. Write **one query** of your own that joins `silver.taxi_trips` to `silver.weather_daily` on the date and returns **trips per day for rainy vs dry days**. Put the result in report §9. This is only a sanity check that the two tables fit together, not an analysis.

> **Git checkpoint:** commit this notebook and your filled report (`git commit -m "notebook 2: silver layer + report"`).

In [0]:
#  Create schema silver and register the three external tables
print(f"Creating schema {CATALOG}.silver...")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

print("Registering silver.taxi_trips...")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.silver.taxi_trips
    USING DELTA
    LOCATION '{BASE}/silver/taxi_trips/'
""")

print("Registering silver.taxi_trips_rejected...")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.silver.taxi_trips_rejected
    USING DELTA
    LOCATION '{BASE}/silver/taxi_trips_rejected/'
""")

print("Registering silver.weather_daily...")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.silver.weather_daily
    USING DELTA
    LOCATION '{BASE}/silver/weather_daily/'
""")

print("\n--- All Silver Tables Registered Successfully ---")

Creating schema dbw_nyc_weather_dev.silver...
Registering silver.taxi_trips...
Registering silver.taxi_trips_rejected...
Registering silver.weather_daily...

--- All Silver Tables Registered Successfully ---


In [0]:
#  Reconciliation Check & Sanity Check Join Query
print("=== RECONCILIATION CHECK ===")
bronze_count = spark.table(f"{CATALOG}.bronze.taxi_trips").count()
silver_count = spark.table(f"{CATALOG}.silver.taxi_trips").count()
rejected_count = spark.table(f"{CATALOG}.silver.taxi_trips_rejected").count()

dupes = globals().get("DUPLICATES_REMOVED", 0)
total_accounted = silver_count + rejected_count + dupes

print(f"Bronze Count:      {bronze_count}")
print(f"Silver Count:      {silver_count}")
print(f"Rejected Count:    {rejected_count}")
print(f"Duplicates Removed:{dupes}")
print(f"Total Accounted:   {total_accounted}")

if bronze_count == total_accounted:
    print("\n✅ RECONCILIATION CHECK PASSED: Bronze == Silver + Rejected + Duplicates!")
else:
    print(f"\n❌ MISMATCH: Difference of {bronze_count - total_accounted} rows.")

print("\n=== SANITY CHECK JOIN: Trips by Weather Condition ===")
query = f"""
    SELECT 
        w.is_rainy,
        COUNT(t.pickup_date) AS total_trips,
        ROUND(COUNT(t.pickup_date) / COUNT(DISTINCT w.weather_date), 2) AS avg_trips_per_day
    FROM {CATALOG}.silver.weather_daily w
    LEFT JOIN {CATALOG}.silver.taxi_trips t
        ON w.weather_date = t.pickup_date
    GROUP BY w.is_rainy
"""
display(spark.sql(query))

=== RECONCILIATION CHECK ===
Bronze Count:      12253805
Silver Count:      10959322
Rejected Count:    1294483
Duplicates Removed:0
Total Accounted:   12253805

✅ RECONCILIATION CHECK PASSED: Bronze == Silver + Rejected + Duplicates!

=== SANITY CHECK JOIN: Trips by Weather Condition ===


is_rainy,total_trips,avg_trips_per_day
false,8268051,118115.01
true,2691271,122330.5


In [0]:
from pyspark.sql import functions as F

_results = []
def check(label, ok):
    _results.append(bool(ok))
    print(("✅" if ok else "❌"), label)

def table_info(name):
    return {r.col_name: r.data_type for r in spark.sql(f"DESCRIBE EXTENDED {name}").collect()}

def summary():
    print(f"\n{sum(_results)} of {len(_results)} checks passed")

EXPECTED_SILVER_TAXI = {
    "vendor_id": "int", "pickup_ts": "timestamp", "dropoff_ts": "timestamp", "passenger_count": "int",
    "trip_distance_miles": "decimal(8,2)", "rate_code_id": "int", "store_and_fwd_flag": "boolean",
    "pickup_location_id": "int", "dropoff_location_id": "int", "payment_type_id": "int",
    "payment_type_desc": "string", "fare_amount": "decimal(10,2)", "extra": "decimal(10,2)",
    "mta_tax": "decimal(10,2)", "tip_amount": "decimal(10,2)", "tolls_amount": "decimal(10,2)",
    "improvement_surcharge": "decimal(10,2)", "total_amount": "decimal(10,2)",
    "congestion_surcharge": "decimal(10,2)", "airport_fee": "decimal(10,2)", "cbd_congestion_fee": "decimal(10,2)",
    "pickup_date": "date", "pickup_hour": "int", "trip_duration_min": "decimal(8,2)",
    "source_month": "string", "silver_loaded_at": "timestamp",
}
EXPECTED_SILVER_WEATHER = {
    "weather_date": "date", "temp_max_c": "decimal(5,1)", "temp_min_c": "decimal(5,1)",
    "precip_mm": "decimal(6,1)", "snowfall_cm": "decimal(6,1)", "wind_max_kmh": "decimal(6,1)",
    "is_rainy": "boolean", "silver_loaded_at": "timestamp",
}
ALLOWED_REASONS = {"pickup_outside_source_month", "dropoff_not_after_pickup", "duration_over_6h",
                   "distance_invalid", "fare_invalid"}

# ---------------- external tables in the right place ----------------
for t in ["taxi_trips", "taxi_trips_rejected", "weather_daily"]:
    info = table_info(f"{CATALOG}.silver.{t}")
    check(f"silver.{t} is EXTERNAL at <container>/silver/{t}",
          info.get("Type") == "EXTERNAL" and info.get("Location", "").rstrip("/").endswith(f"/silver/{t}"))

# ---------------- silver taxi: contract ----------------
st = spark.table(f"{CATALOG}.silver.taxi_trips")
check("silver.taxi_trips has exactly the agreed columns and types", dict(st.dtypes) == EXPECTED_SILVER_TAXI)
stats = st.agg(
    F.min("trip_duration_min").alias("min_dur"), F.max("trip_duration_min").alias("max_dur"),
    F.min("fare_amount").alias("min_fare"), F.max("fare_amount").alias("max_fare"),
    F.min("trip_distance_miles").alias("min_dist"), F.max("trip_distance_miles").alias("max_dist"),
    F.min("passenger_count").alias("min_pax"), F.max("passenger_count").alias("max_pax"),
    F.sum((F.date_format("pickup_ts", "yyyy-MM") != F.col("source_month")).cast("int")).alias("wrong_month"),
).first()
check("no trip with duration <= 0 or > 360 min", stats.min_dur > 0 and stats.max_dur <= 360)
check("no fare <= 0 or >= 500", stats.min_fare > 0 and stats.max_fare < 500)
check("no distance <= 0 or >= 100", stats.min_dist > 0 and stats.max_dist < 100)
check("passenger_count is NULL or between 1 and 6", stats.min_pax >= 1 and stats.max_pax <= 6)
check("every pickup falls in its own source_month", stats.wrong_month == 0)

# ---------------- rejected table ----------------
rj = spark.table(f"{CATALOG}.silver.taxi_trips_rejected")
check("rejected table has reject_reason and rejected_at", {"reject_reason", "rejected_at"} <= set(rj.columns))
reasons = {r.reject_reason for r in rj.select("reject_reason").distinct().collect()}
check("every reject_reason is one of the 5 agreed strings", reasons <= ALLOWED_REASONS)

# ---------------- reconciliation: nothing disappears silently ----------------
b_n, s_n, r_n = spark.table(f"{CATALOG}.bronze.taxi_trips").count(), st.count(), rj.count()
check(f"bronze ({b_n:,}) == silver ({s_n:,}) + rejected ({r_n:,}) + duplicates removed ({DUPLICATES_REMOVED:,})",
      b_n == s_n + r_n + DUPLICATES_REMOVED)

# ---------------- silver weather ----------------
sw = spark.table(f"{CATALOG}.silver.weather_daily")
check("silver.weather_daily has the agreed columns and types", dict(sw.dtypes) == EXPECTED_SILVER_WEATHER)
check("weather: 92 rows, one per date", sw.count() == 92 and sw.select("weather_date").distinct().count() == 92)

summary()
print(f"Bronze {b_n:,} | Silver {s_n:,} | Rejected {r_n:,} | Duplicates removed {DUPLICATES_REMOVED:,}")

✅ silver.taxi_trips is EXTERNAL at <container>/silver/taxi_trips
✅ silver.taxi_trips_rejected is EXTERNAL at <container>/silver/taxi_trips_rejected
✅ silver.weather_daily is EXTERNAL at <container>/silver/weather_daily
✅ silver.taxi_trips has exactly the agreed columns and types
✅ no trip with duration <= 0 or > 360 min
✅ no fare <= 0 or >= 500
✅ no distance <= 0 or >= 100
✅ passenger_count is NULL or between 1 and 6
✅ every pickup falls in its own source_month
✅ rejected table has reject_reason and rejected_at
✅ every reject_reason is one of the 5 agreed strings
✅ bronze (12,253,805) == silver (10,959,322) + rejected (1,294,483) + duplicates removed (0)
✅ silver.weather_daily has the agreed columns and types
✅ weather: 92 rows, one per date

14 of 14 checks passed
Bronze 12,253,805 | Silver 10,959,322 | Rejected 1,294,483 | Duplicates removed 0
